# L5–L6 核方法

## 正则化、非线性特征、SVM、RKHS、核回归与局部平均

本笔记将原有 L5 笔记、L6 内容以及以下参考资料中的相关补充整合为一套连续课程笔记：

- L5.pdf 与 L6.pdf；
- Francis Bach，Learning Theory from First Principles，第 3、4、6、7、12 章；
- Understanding Machine Learning: Solution Manual，第 13、15、16、22 章。

每个编号 Section 都是独立的 Markdown cell。除非特别说明，统一采用平均经验风险记号：

$$
\widehat R_n(f)=\frac1n\sum_{i=1}^n\ell(y_i,f(x_i)).
$$

如果损失函数或正则项前是否带 \(1/2\) 的约定不同，只会重新缩放 \(\lambda\)，不会改变估计器的基本结构。

## Section 1 — 线性预测、最小二乘与两种设计假设

模型所谓“线性”，指的是它对参数线性：

$$
f_\theta(x)=\phi(x)^\top\theta.
$$

它不一定对原始输入 \(x\) 线性。特征映射 \(\phi(x)\) 可以包含多项式、三角函数、文本特征，甚至无限多个坐标。

给定设计矩阵 \(\Phi\in\mathbb R^{n\times d}\) 和标签 \(y\in\mathbb R^n\)，普通最小二乘求解

$$
\min_\theta\frac1n\|y-\Phi\theta\|_2^2.
$$

若 \(\Phi\) 满列秩，则

$$
\widehat\theta_{\mathrm{OLS}}
=(\Phi^\top\Phi)^{-1}\Phi^\top y,
\qquad
\widehat y=P_\Phi y,
$$

其中

$$
P_\Phi
=\Phi(\Phi^\top\Phi)^{-1}\Phi^\top
$$

是到 \(\operatorname{im}(\Phi)\) 的正交投影。因此残差满足

$$
\Phi^\top(y-\widehat y)=0.
$$

显式逆矩阵主要用于理论推导。数值计算中应优先使用 QR 分解、Cholesky 分解、共轭梯度或梯度方法。

必须区分两种风险设定：

- 固定设计：输入被视为确定量，只重复生成响应。这主要是训练输入位置上的去噪问题。
- 随机设计：\(X\sim p_X\)，风险在新的随机输入上评估。这才是通常意义上的泛化。

在正确设定、满秩的固定设计模型中，若噪声方差为 \(\sigma^2\)，则

$$
\mathbb E[R(\widehat\theta_{\mathrm{OLS}})]-R^*
=\frac{\sigma^2d}{n}.
$$

若在同一批设计点上重新生成测试响应，则

$$
\mathbb E[\mathrm{MSE}_{\mathrm{train}}]
=\sigma^2\left(1-\frac dn\right),
$$

$$
\mathbb E[\mathrm{MSE}_{\mathrm{test}}]
=\sigma^2\left(1+\frac dn\right).
$$

因此训练误差平均会以 \(2\sigma^2d/n\) 的幅度乐观地低估测试误差。

## Section 2 — 岭回归：目标函数、primal 与 dual

岭回归加入 \(\ell_2\) 正则项：

$$
\widehat\theta_\lambda
\in\arg\min_\theta
\left\{
\frac1n\|y-\Phi\theta\|_2^2
+\lambda\|\theta\|_2^2
\right\}.
$$

正规方程为

$$
(\Phi^\top\Phi+n\lambda I_d)\widehat\theta_\lambda
=\Phi^\top y,
$$

所以

$$
\boxed{
\widehat\theta_\lambda
=(\Phi^\top\Phi+n\lambda I_d)^{-1}\Phi^\top y.
}
$$

利用矩阵求逆恒等式，还可以写成

$$
\boxed{
\widehat\theta_\lambda
=\Phi^\top(\Phi\Phi^\top+n\lambda I_n)^{-1}y.
}
$$

两种形式决定计算方式：

- \(d\ll n\) 时，求解 \(d\times d\) 的 primal 系统；
- \(n\ll d\) 时，求解 \(n\times n\) 的 dual 系统；
- 在无限维特征空间中，只要能够计算内积，dual 形式仍然可用。

加入 \(\lambda I\) 会把协方差矩阵的每个特征值向上平移：

$$
\mu_j(\widehat\Sigma+\lambda I)
=\mu_j(\widehat\Sigma)+\lambda,
\qquad
\widehat\Sigma=\frac1n\Phi^\top\Phi.
$$

因此 Ridge 改善条件数，并防止数据支持很弱的方向产生极大系数。

若噪声为 Gaussian，且参数先验为

$$
\theta\sim
\mathcal N\left(
0,\frac{\sigma^2}{n\lambda}I
\right),
$$

则岭回归同时是后验均值和 MAP 估计。

## Section 3 — 偏差—方差与近似—估计分解

设

$$
Y=f_*(X)+\varepsilon,
$$

其中噪声独立、均值为零。随机预测器 \(\widehat f\) 的平方预测误差可分解为

$$
\boxed{
\mathbb E[(Y-\widehat f(X))^2]
=
\sigma^2
+
\left(
f_*(X)-\mathbb E[\widehat f(X)]
\right)^2
+
\operatorname{Var}(\widehat f(X)).
}
$$

三项分别是不可约噪声、偏差平方和方差。

这与学习理论中的误差分解不是同一个公式。若

$$
f_{\mathcal F}^*
\in\arg\min_{f\in\mathcal F}R(f),
$$

则

$$
R(\widehat f)-R^*
=
\underbrace{
R(f_{\mathcal F}^*)-R^*
}_{\text{近似误差}}
+
\underbrace{
R(\widehat f)-R(f_{\mathcal F}^*)
}_{\text{估计误差}}.
$$

扩大模型容量通常会降低近似误差，却提高估计误差；增强正则化通常会提高偏差或近似误差，却降低方差或估计误差。

Gauss–Markov 定理说明，在经典线性模型假设下，OLS 是最佳线性无偏估计器。这里“无偏”是关键限制。Ridge 虽然有偏，却可能因为大幅降低方差而获得更小的预测风险。

## Section 4 — Ridge 的谱收缩与有效自由度

设

$$
\widehat\Sigma
=
U\operatorname{diag}(\mu_1,\ldots,\mu_d)U^\top,
\qquad
a_j=u_j^\top\theta_*.
$$

在正确设定的固定设计模型中，

$$
\begin{aligned}
\mathbb E[R(\widehat\theta_\lambda)]-R^*
={}&
\lambda^2\theta_*^\top
(\widehat\Sigma+\lambda I)^{-2}
\widehat\Sigma\theta_*\\
&+
\frac{\sigma^2}{n}
\operatorname{tr}
\left[
\widehat\Sigma^2
(\widehat\Sigma+\lambda I)^{-2}
\right].
\end{aligned}
$$

在协方差特征基中，

$$
\operatorname{Bias}^2
=
\sum_j
\frac{\lambda^2\mu_j}{(\mu_j+\lambda)^2}a_j^2,
$$

$$
\operatorname{Variance}
=
\frac{\sigma^2}{n}
\sum_j
\left(
\frac{\mu_j}{\mu_j+\lambda}
\right)^2.
$$

每个方向的收缩乘子是

$$
s_j(\lambda)
=
\frac{\mu_j}{\mu_j+\lambda}.
$$

- \(\mu_j\gg\lambda\)：该方向得到充分数据支持，基本保留；
- \(\mu_j\ll\lambda\)：该方向很不稳定，被强烈抑制。

需要区分两种相关的有效维数：

$$
df_1(\lambda)
=
\sum_j\frac{\mu_j}{\mu_j+\lambda}
=\operatorname{tr}(H_\lambda),
$$

$$
df_2(\lambda)
=
\sum_j
\left(
\frac{\mu_j}{\mu_j+\lambda}
\right)^2
=\operatorname{tr}(H_\lambda^2).
$$

\(df_1\) 是软参数数目；\(df_2\) 直接控制固定设计下的方差。

PCA 后接 OLS 相当于硬谱截断；Ridge 相当于软谱收缩。

## Section 5 — 正则化为什么能改善泛化

正则化通过四个相互联系的机制发挥作用：

1. 改善条件数：消除优化问题中接近平坦的方向。
2. 控制容量：限制预测函数的范数。
3. 提高稳定性：替换一个训练样本时，解只发生较小变化。
4. 编码先验：规定哪些解应当被视为简单。

若损失对预测是 \(G\)-Lipschitz，且 \(\|\phi(x)\|\le R\)，则 \(\ell_2\) 正则化算法的期望稳定性项典型为

$$
O\left(
\frac{G^2R^2}{\lambda n}
\right).
$$

若算法具有 \(\epsilon_{\mathrm{stab}}\) 稳定性，并且是 \(\epsilon_{\mathrm{ERM}}\)-近似经验风险最小化器，则

$$
\boxed{
\mathbb E[
R(A(S))-R(f^*)
]
\le
\epsilon_{\mathrm{stab}}
+\epsilon_{\mathrm{ERM}}.
}
$$

因此增大 \(\lambda\) 会提高稳定性，但也会增加正则化偏差。

若从 \(M\) 个候选模型或超参数中选择一个，模型选择通常要付出

$$
\sqrt{\frac{\log M}{n}}
$$

量级的复杂度代价。验证集或结构风险最小化就是在显式处理这一代价。验证集不能过小，否则 \(\sqrt{\log M/n_{\mathrm{val}}}\) 会过大。

## Section 6 — 超越 Ridge 的正则化

一般的正则化经验风险最小化问题为

$$
\min_\theta
\left\{
\widehat R_n(\theta)+\lambda\Omega(\theta)
\right\}.
$$

常见选择包括：

- \(\Omega(\theta)=\|\theta\|_2^2\)：鼓励小而稠密的系数；
- \(\Omega(\theta)=\|\theta\|_1\)：鼓励稀疏系数；
- \(\Omega(\theta)=\theta^\top M\theta\)，其中 \(M\succ0\)：方向相关的 Tikhonov 正则化；
- 约束 \(\|\theta\|\le D\)：用可行域控制容量。

对一维问题

$$
\min_w
\left\{
\frac12w^2-xw+\lambda|w|
\right\},
$$

解是软阈值

$$
\boxed{
w
=
\operatorname{sign}(x)(|x|-\lambda)_+.
}
$$

这说明 \(\ell_1\) 正则化能够把系数精确压缩到零；\(\ell_2\) 通常只能连续缩小系数。

正则化也可以是隐式的。初始化、梯度下降、早停、网络结构、数据增强以及随机性，都可能在多个经验风险最小解中选择某一个解。

## Section 7 — 非线性特征映射

将线性方法应用于非线性特征，就可以得到非线性预测：

$$
h(x)
=
\operatorname{sign}
\left(
w^\top\phi(x)+b
\right).
$$

例如二维二次特征

$$
\phi(x_1,x_2)
=
(x_1,x_2,x_1^2,x_1x_2,x_2^2)
$$

会把特征空间中的超平面变成原输入空间中的二次曲线。

对 \(d\) 个变量，总次数恰好为 \(s\) 的单项式个数为

$$
\binom{d+s-1}{s}.
$$

次数不超过 \(s\)、包含常数项的单项式个数为

$$
\binom{d+s}{s}.
$$

任意有限假设类都能被线性化。若

$$
\mathcal H=\{h_1,\ldots,h_M\},
$$

可以为每个假设的输出建立一个指示特征，再加常数坐标。此时每个 \(h_j\) 都能由类似 \((2e_j,-1)\) 的线性参数表示。

真正困难的不是这种特征映射是否存在，而是显式特征维数可能极大。手工特征、隐式核特征和神经网络学习特征，是选择归纳偏置的三种不同方式。

## Section 8 — SVM 的几何与间隔

对超平面

$$
w^\top x+b=0,
$$

点 \(x\) 到超平面的欧氏距离为

$$
\frac{|w^\top x+b|}{\|w\|}.
$$

在线性可分数据上，最大化最小几何间隔等价于 hard-margin SVM：

$$
\min_{w,b}\frac12\|w\|^2
$$

满足

$$
y_i(w^\top x_i+b)\ge1.
$$

之所以可以把函数间隔规范化为 1，是因为同时把 \(w,b\) 乘以正数不会改变分类结果。

若 \(\|x_i\|\le\rho\)，并且存在单位向量以间隔 \(\gamma\) 分离数据，则感知机的犯错次数满足

$$
\boxed{
T_{\mathrm{mistakes}}
\le
\left(
\frac{\rho}{\gamma}
\right)^2.
}
$$

因此间隔既是几何复杂度，也是算法收敛速度的量。

## Section 9 — 软间隔 SVM、hinge loss、对偶与支持向量

软间隔 primal 问题为

$$
\min_{w,b,\xi}
\frac12\|w\|^2
+C\sum_i\xi_i
$$

满足

$$
y_i(w^\top x_i+b)\ge1-\xi_i,
\qquad
\xi_i\ge0.
$$

消去松弛变量后，

$$
\min_{w,b}
\frac1n
\sum_i
(1-y_i(w^\top x_i+b))_+
+\frac\lambda2\|w\|^2,
$$

其中

$$
\boxed{
\lambda=\frac1{nC}.
}
$$

核化后的对偶问题为

$$
\max_\alpha
\sum_i\alpha_i
-\frac12
\sum_{i,j}
\alpha_i\alpha_jy_iy_jK(x_i,x_j)
$$

满足

$$
0\le\alpha_i\le C,
\qquad
\sum_i\alpha_i y_i=0.
$$

最优参数满足

$$
w=\sum_i\alpha_i y_i\phi(x_i).
$$

KKT 条件给出：

- margin \(>1\)：\(\alpha_i=0\)；
- margin \(=1\)：通常 \(0<\alpha_i<C\)；
- 位于间隔内或误分类：通常 \(\alpha_i=C\)。

若 \(\|x\|\le R\)，hinge loss 对 \(w\) 是 \(R\)-Lipschitz：

$$
|\ell(w_1)-\ell(w_2)|
\le
R\|w_1-w_2\|.
$$

即使数据线性可分，有限 \(C\) 的 soft SVM 也不一定等于 hard SVM：违反一个极难处理的点，可能比使用极大的参数范数更便宜。

## Section 10 — 核技巧与核化线性算法

设特征映射 \(\phi:\mathcal X\to\mathcal H\) 满足

$$
k(x,z)
=
\langle\phi(x),\phi(z)\rangle_{\mathcal H}.
$$

若一个算法只通过内积使用特征向量，就可以用核函数代替特征空间内积。

SVM 的预测为

$$
h(x)
=
\operatorname{sign}
\left(
\sum_i\alpha_i y_i k(x_i,x)+b
\right).
$$

整个计算过程不需要显式构造特征向量。

核感知机可以使用有符号系数：

$$
w^{(t)}
=
\sum_i\alpha_i^{(t)}\phi(x_i).
$$

若样本 \(i\) 被误分类，

$$
y_i\sum_j\alpha_jK(x_j,x_i)\le0,
$$

并更新

$$
\alpha_i\leftarrow\alpha_i+y_i.
$$

预测为

$$
\operatorname{sign}
\left(
\sum_i\alpha_iK(x_i,x)
\right).
$$

另一种记号使用非负犯错次数 \(c_i\)，写成 \(w=\sum_i c_i y_i\phi(x_i)\)。两套记号等价，但不能混用。

最近质心、PCA 以及很多只依赖点积的算法也可以核化。

## Section 11 — 合法核与 Gram 矩阵

实值对称函数 \(k\) 是合法正半定核，当且仅当对任意有限点集 \(x_1,\ldots,x_n\)，Gram 矩阵

$$
K_{ij}=k(x_i,x_j)
$$

都满足

$$
c^\top Kc\ge0,
\qquad
\forall c\in\mathbb R^n.
$$

以下两种说法等价：

1. 所有 Gram 矩阵都为正半定；
2. 存在某个 Hilbert 空间特征映射，使
   \(k(x,z)=\langle\phi(x),\phi(z)\rangle\)。

常用封闭规则包括：

- 若 \(k_1,k_2\) 是核且 \(a,b\ge0\)，则 \(ak_1+bk_2\) 是核；
- \(k_1k_2\) 是核；
- 对任意函数 \(g\)，\(g(x)k(x,z)g(z)\) 是核；
- 核函数的逐点极限在极限存在时仍是核。

两个直观例子是

$$
k(i,j)=\min(i,j),
$$

它对应前缀指示特征；以及

$$
k(E,E')=|E\cap E'|,
$$

它对应集合指示向量。

一个函数即使处处非负，也不一定是合法核。正半定性是对整个 Gram 矩阵二次型的要求。

## Section 12 — 重要核族与多项式特征维数

线性核为

$$
k(x,z)=x^\top z.
$$

齐次多项式核为

$$
k_s(x,z)=(x^\top z)^s.
$$

对多重指标 \(\alpha\in\mathbb N^d\)、\(|\alpha|=s\)，一个规范化特征为

$$
\phi_\alpha(x)
=
\sqrt{
\frac{s!}{\alpha_1!\cdots\alpha_d!}
}
x^\alpha.
$$

最小显式维数为

$$
\binom{d+s-1}{s}.
$$

非齐次多项式核

$$
k_s(x,z)
=(c+x^\top z)^s,
\qquad c\ge0,
$$

在 \(c>0\) 时包含所有不超过 \(s\) 次的单项式，维数为

$$
\binom{d+s}{s}.
$$

Gaussian RBF 核为

$$
k(x,z)
=
\exp\left(
-\frac{\|x-z\|^2}{2\sigma^2}
\right)
=
\exp(-\gamma\|x-z\|^2).
$$

Laplacian 核为

$$
k(x,z)
=
\exp\left(
-\frac{\|x-z\|}{r}
\right).
$$

Matérn 核的参数可以控制有限阶光滑性。还可以为集合、词袋、序列、字符串、树、图、点云和概率分布设计专门的核。

## Section 13 — 平移不变核与 Fourier 视角

设

$$
k(x,z)=q(x-z).
$$

Bochner 定理说明：连续平移不变函数是正半定核，当且仅当它是某个有限非负测度的 Fourier 变换。若存在谱密度，则

$$
\widehat q(\omega)\ge0.
$$

相应 RKHS 范数具有频域形式

$$
\boxed{
\|f\|_{\mathcal H}^2
=
\frac1{(2\pi)^d}
\int_{\mathbb R^d}
\frac{|\widehat f(\omega)|^2}
{\widehat q(\omega)}
\,d\omega.
}
$$

因此核规定了每个频率的代价：

- \(\widehat q(\omega)\) 大：该频率代价低；
- \(\widehat q(\omega)\) 小：该频率受到强烈惩罚。

Fourier 域中乘以 \(\omega_j\) 对应输入空间中的求导，因此频谱衰减决定函数光滑性：

- Laplacian 或 exponential 核对应有限阶 Sobolev 控制；
- Matérn 核可以显式选择有限光滑阶数；
- Gaussian 核极强地惩罚高频，产生更小、更光滑的 RKHS。

实践中常把核尺度 \(r\) 取为训练样本两两距离的某个分位数，例如中位数。但这只是启发式选择，通常仍需验证。

## Section 14 — RKHS 构造与再生性质

从合法核出发，先考虑有限展开

$$
\mathcal H_0
=
\left\{
\sum_i\alpha_i k(\cdot,x_i)
\right\}.
$$

定义内积

$$
\left\langle
\sum_i\alpha_i k(\cdot,x_i),
\sum_j\beta_j k(\cdot,z_j)
\right\rangle_{\mathcal H}
=
\sum_{i,j}
\alpha_i\beta_jk(x_i,z_j).
$$

对该空间取 Hilbert 完备化，就得到核生成的规范 RKHS。

它的核心再生性质是

$$
\boxed{
\langle k(\cdot,x),f\rangle_{\mathcal H}
=f(x).
}
$$

由 Cauchy–Schwarz，

$$
|f(x)|
\le
\|f\|_{\mathcal H}
\sqrt{k(x,x)}.
$$

若

$$
\sup_xk(x,x)\le R^2,
$$

则

$$
\|f\|_\infty
\le
R\|f\|_{\mathcal H}.
$$

并非所有函数 Hilbert 空间都是 RKHS。普通 \(L^2\) 中点值评价不连续，而且函数只在“几乎处处相等”的意义下定义。

特征映射并不唯一：可以旋转坐标、复制坐标或嵌入更大的空间。但核生成的规范 RKHS 在等距同构意义下是唯一的。

## Section 15 — 表示定理与最小范数插值

考虑只通过训练点函数值和 RKHS 范数访问 \(f\) 的目标：

$$
\Psi
\left(
f(x_1),\ldots,f(x_n),
\|f\|_{\mathcal H}^2
\right).
$$

若 \(\Psi\) 对最后的范数变量严格递增，则每个最优解都位于训练核函数张成空间中：

$$
\boxed{
f(\cdot)
=
\sum_{i=1}^n
\alpha_i k(\cdot,x_i).
}
$$

证明把函数正交分解为

$$
f=f_D+f_\perp,
$$

其中 \(f_D\) 位于 \(k(\cdot,x_i)\) 的张成空间。正交分量满足

$$
f_\perp(x_i)=0,
$$

所以它不改变任何训练预测，却会增大范数，因此最优解不能含有该分量。

这个几何结论本身不要求损失凸。凸性主要影响优化和唯一性。

若精确插值可行，则最小 RKHS 范数插值器满足

$$
f(\cdot)
=
\sum_i\alpha_i k(\cdot,x_i),
\qquad
K\alpha=y.
$$

若 \(K\) 奇异，最小范数系数代表可写为 \(K^\dagger y\)。在系数中加入零空间分量不会改变预测函数。

## Section 16 — 核岭回归

核岭回归求解

$$
\min_{f\in\mathcal H}
\left\{
\frac1n
\sum_{i=1}^n
(y_i-f(x_i))^2
+\lambda\|f\|_{\mathcal H}^2
\right\}.
$$

由表示定理，

$$
\widehat f_\lambda(x)
=
\sum_i\alpha_i k(x,x_i),
$$

一个规范系数解为

$$
\boxed{
\alpha=(K+n\lambda I)^{-1}y.
}
$$

若 \(K\) 奇异，展开系数可能不唯一；但当 \(\lambda>0\) 时预测函数是唯一的。上式给出一个规范代表。

定义

$$
q(x)
=
(k(x,x_1),\ldots,k(x,x_n))^\top.
$$

KRR 对标签向量是线性的：

$$
\widehat f_\lambda(x)
=
q(x)^\top
(K+n\lambda I)^{-1}y
=
\sum_iw_i(x)y_i.
$$

在训练点上，

$$
\widehat y
=
H_\lambda y,
\qquad
H_\lambda
=
K(K+n\lambda I)^{-1}.
$$

与局部平均不同，KRR 权重可以为负，也不一定和为 1。负权重可以形成高阶抵消，这是 KRR 能够适应更光滑目标函数的重要原因。

## Section 17 — KRR 的统计分析与有效维数

在随机设计下，定义总体协方差算子

$$
\Sigma
=
\mathbb E[
\phi(X)\otimes\phi(X)
].
$$

它把 RKHS 几何与预测误差连接起来：

$$
\|g\|_{L^2(p)}^2
=
\langle g,\Sigma g\rangle_{\mathcal H}.
$$

总体有效维数为

$$
\boxed{
\mathcal N(\lambda)
=
\operatorname{tr}
\left[
\Sigma(\Sigma+\lambda I)^{-1}
\right].
}
$$

对可能不属于 RKHS 的目标函数，定义

$$
A(\lambda,f_*)
=
\inf_{f\in\mathcal H}
\left\{
\|f-f_*\|_{L^2(p)}^2
+\lambda\|f\|_{\mathcal H}^2
\right\}.
$$

忽略技术性的乘法常数，KRR 风险具有结构

$$
\mathbb E
\|\widehat f_\lambda-f_*\|_{L^2(p)}^2
\lesssim
\frac{\sigma^2}{n}\mathcal N(\lambda)
+A(\lambda,f_*).
$$

若协方差特征值满足

$$
\mu_j\asymp j^{-2s/d},
$$

则

$$
\mathcal N(\lambda)
\asymp
\lambda^{-d/(2s)}.
$$

若目标具有 \(t\) 阶 Sobolev 光滑性，则逼近项典型为 \(\lambda^{t/s}\)。平衡偏差和方差得到

$$
\lambda_{\mathrm{opt}}
\asymp
n^{-2s/(2t+d)},
$$

以及

$$
\boxed{
\operatorname{excess\ risk}
\asymp
n^{-2t/(2t+d)}.
}
$$

当 \(t=1\) 时得到局部平均的 \(n^{-2/(d+2)}\)；目标越光滑，KRR 可以获得越快的收敛率。这就是 KRR 的光滑度自适应。

## Section 18 — 可扩展核计算

精确核矩阵需要 \(O(n^2)\) 存储，直接求解 KRR 通常需要 \(O(n^3)\) 时间。

直接在系数 \(\alpha\) 上优化也可能条件很差，因为 Hessian 含有 \(K\) 的因子。若

$$
K=\Phi\Phi^\top,
$$

改用显式特征坐标后，Hessian 为

$$
\frac1n\Phi^\top\Phi+\lambda I,
$$

其最小特征值至少为 \(\lambda\)。

Nyström 近似选取 \(m\) 个 landmark 列 \(I\)：

$$
K
\approx
K(:,I)K(I,I)^{-1}K(I,:).
$$

它产生 \(m\) 维、依赖训练数据的近似特征，典型计算量为 \(O(nm^2)\)。

随机特征利用

$$
k(x,z)
=
\mathbb E_v[
\varphi(x,v)\varphi(z,v)
]
$$

并近似为

$$
\widetilde k(x,z)
=
\frac1m
\sum_{j=1}^m
\varphi(x,v_j)\varphi(z,v_j).
$$

对平移不变核，随机 Fourier 特征可使用

$$
\sqrt2
\cos(\omega^\top x+b),
$$

其中 \(\omega\) 从规范化谱测度采样，\(b\) 在 \([0,2\pi]\) 上均匀采样。

Nyström 是依赖数据的降维；随机特征可以在观察训练输入之前生成。

## Section 19 — 无正则插值、隐式偏置与双下降

当 \(d>n\) 且 \(XX^\top\) 可逆时，

$$
X\theta=y
$$

有无穷多个插值解。从

$$
\theta_0=0
$$

开始的梯度下降始终位于 \(X\) 的行空间中，并收敛到

$$
\boxed{
\theta^\dagger
=
X^\top(XX^\top)^{-1}y.
}
$$

这是最小 \(\ell_2\) 范数插值解。RKHS 中对应最小 RKHS 范数插值函数。

对线性可分 logistic 回归，参数范数会趋于无穷，但归一化方向收敛到 hard-margin SVM 的方向。这是另一种隐式正则化。

隐式偏置并不一定有利。它依赖参数化、初始化、优化器和数据结构。如果真实参数稀疏，隐式 \(\ell_2\) 偏好可能不如显式 \(\ell_1\) 正则化。

在各向同性 Gaussian 线性模型中，最小范数估计器具有精确期望超额风险：

$$
\mathbb E[
R(\widehat\theta)-R^*
]
=
\frac{\sigma^2d}{n-d-1},
\qquad
d\le n-2,
$$

以及

$$
\mathbb E[
R(\widehat\theta)-R^*
]
=
\frac{\sigma^2n}{d-n-1}
+\|\theta_*\|^2\frac{d-n}{d},
\qquad
d\ge n+2.
$$

在插值阈值 \(d=n\) 附近方差爆炸，进入过参数化区域后又下降，从而形成双下降。适当调节 Ridge 的 \(\lambda\) 可以平滑甚至消除该峰值。

## Section 20 — KDE 与局部平均

密度估计或局部平均中的平滑核，与 RKHS 核不是同一种数学对象。

在 \(\mathbb R^d\) 中，核密度估计为

$$
\widehat p_h(x)
=
\frac1{nh^d}
\sum_{i=1}^n
q\left(
\frac{x-x_i}{h}
\right),
$$

其中通常要求

$$
q\ge0,
\qquad
\int q=1.
$$

这里 \(h^{-d}\) 对密度归一化是必需的。

带宽控制偏差—方差：

- \(h\) 小：估计更局部、偏差低，但噪声大；
- \(h\) 大：估计更稳定，但容易过度平滑。

对于一维二阶光滑密度，经典积分平方误差具有

$$
\operatorname{Variance}
=
O\left(
\frac1{nh}
\right),
\qquad
\operatorname{Bias}^2
=
O(h^4),
$$

因此

$$
h_{\mathrm{opt}}
\asymp
n^{-1/5}.
$$

这个结论不能与只假设 Lipschitz 的 Nadaraya–Watson 回归带宽率混用。

## Section 21 — Nadaraya–Watson 回归与 Attention

Nadaraya–Watson 回归使用归一化非负权重：

$$
\widehat f_h(x)
=
\sum_iw_i(x)y_i,
$$

$$
w_i(x)
=
\frac{q_h(x-x_i)}
{\sum_jq_h(x-x_j)}.
$$

这里 \(q_h\) 只需要逐点非负，不需要满足 Gram 正半定。

条件于训练输入，

$$
\begin{aligned}
\mathbb E[
(\widehat f_h(x)-f_*(x))^2
\mid X_{1:n}
]
={}&
\left[
\sum_i
w_i(x)
(f_*(x_i)-f_*(x))
\right]^2\\
&+
\sum_i
w_i(x)^2
\operatorname{Var}(Y_i\mid X_i).
\end{aligned}
$$

对 \(d\) 维 \(B\)-Lipschitz 目标，

$$
\operatorname{Bias}^2
\asymp
B^2h^2,
\qquad
\operatorname{Variance}
\asymp
\frac{\sigma^2}{nh^d}.
$$

所以

$$
\boxed{
h_{\mathrm{opt}}
\asymp
n^{-1/(d+2)},
\qquad
\operatorname{risk}
\asymp
n^{-2/(d+2)}.
}
$$

一致性要求

$$
h\to0,
\qquad
nh^d\to\infty.
$$

若目标二阶光滑、核具有适当对称性，则平方偏差变成 \(O(h^4)\)，风险率改善为 \(n^{-4/(d+4)}\)。

缩放点积 Attention 为

$$
\operatorname{Attention}(Q,K,V)
=
\operatorname{softmax}
\left(
\frac{QK^\top}{\sqrt d}
\right)V.
$$

它在形式上类似 NW：每个 query 都产生一组对 value 的归一化非负权重。但这只是结构类比：

- query、key 和 value 的映射由学习得到；
- 分数矩阵不一定对称；
- 分数不必构成正半定 Gram 矩阵；
- Attention 不会自动成为 RKHS 方法。

## Section 22 — 统一比较、实践流程与复习问题

三类主要加权估计器可比较如下：

| 方法 | 相似度条件 | 权重性质 | 主要调节量 |
|---|---|---|---|
| Nadaraya–Watson | 平滑窗逐点非负 | 非负且和为 1 | 带宽 \(h\) |
| 核岭回归 | Gram 核正半定 | 可为负，通常不和为 1 | 正则化 \(\lambda\) |
| Attention | 学习得到的 softmax 分数 | 非负，每行和为 1 | 学习得到的 \(Q,K,V\) 与温度尺度 |

一个实用的核方法工作流程是：

1. 明确预测任务，并判断线性模型是否合理。
2. 标准化输入，选择特征映射或核函数族。
3. 检查核是否合法，并理解各超参数的意义。
4. 用验证集或交叉验证调节核尺度与正则化。
5. 检查条件数、有效自由度和支持向量数量。
6. 当 \(n\) 很大时，使用 Nyström、随机特征或迭代求解器。
7. 明确理论分析属于固定设计还是随机设计。
8. 把无正则插值理解为依赖算法的最小范数选择，而不是所有插值器的共同性质。

复习问题：

1. 为什么 Ridge 优于 OLS 不违背 Gauss–Markov 定理？
2. 推导岭回归的 primal 与 dual 公式。
3. 解释 \(df_1(\lambda)\) 和 \(df_2(\lambda)\) 的区别。
4. 为什么可分数据上的有限 \(C\) soft SVM 不一定等于 hard SVM？
5. 证明 \(k(E,E')=|E\cap E'|\) 是合法核。
6. 使用正交分解证明表示定理。
7. 为什么 KRR 权重可以为负，而 NW 权重不能？
8. 解释一致性条件 \(h\to0\) 和 \(nh^d\to\infty\)。
9. 解释梯度下降如何选择最小范数插值解。
10. 区分平滑核、Mercer 核和 Attention 分数。